In [1]:
import requests
import io
import zipfile
import tempfile
import os
import pandas as pd
from dbfread import DBF

In [72]:
def cargar_url_en_df(url: str, nombre: str) -> pd.DataFrame:
    """
    Descarga un archivo desde una URL directamente en memoria
    y lo retorna como DataFrame.
    Soporta: zip + (dbf, xlsx, xls, csv)
    """
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, allow_redirects=True, timeout=60)
    response.raise_for_status()
    print(f"[{nombre}] Descargado: {len(response.content)/1024/1024:.2f} MB | Content-Type: {response.headers.get('Content-Type','?')}")

    content = response.content
    buf = io.BytesIO(content)

    # --- ZIP (puede contener DBF, xlsx u otros) ---
    if zipfile.is_zipfile(io.BytesIO(content)):
        with zipfile.ZipFile(buf) as z:
            nombres = z.namelist()
            print(f"[{nombre}] Archivos en ZIP: {nombres}")

            # Buscar DBF dentro del ZIP
            dbf_files = [n for n in nombres if n.lower().endswith(".dbf")]
            xlsx_files = [n for n in nombres if n.lower().endswith(".xlsx")]
            xls_files  = [n for n in nombres if n.lower().endswith(".xls")]
            csv_files  = [n for n in nombres if n.lower().endswith(".csv")]

            if dbf_files:
                dbf_bytes = z.read(dbf_files[0])
                with tempfile.NamedTemporaryFile(suffix=".dbf", delete=False) as tmp:
                    tmp.write(dbf_bytes)
                    tmp_path = tmp.name
                try:
                    tabla = DBF(tmp_path, encoding="latin-1", ignore_missing_memofile=True)
                    df = pd.DataFrame(iter(tabla))
                finally:
                    os.unlink(tmp_path)

            elif xlsx_files:
                df = pd.read_excel(io.BytesIO(z.read(xlsx_files[0])), engine="openpyxl")
            elif xls_files:
                df = pd.read_excel(io.BytesIO(z.read(xls_files[0])), engine="xlrd")
            elif csv_files:
                raw = z.read(csv_files[0])
                try:
                    df = pd.read_csv(io.BytesIO(raw), encoding="utf-8")
                except UnicodeDecodeError:
                    df = pd.read_csv(io.BytesIO(raw), encoding="latin-1")
            else:
                raise ValueError(f"[{nombre}] ZIP no contiene DBF, xlsx, xls ni csv. Archivos: {nombres}")

    # --- Excel directo ---
    elif content[0:8] == b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1":
        df = pd.read_excel(buf, engine="xlrd")

    # --- CSV directo ---
    else:
        try:
            df = pd.read_csv(buf, encoding="utf-8", sep=None, engine="python")
        except UnicodeDecodeError:
            df = pd.read_csv(io.BytesIO(content), encoding="latin-1", sep=None, engine="python")

    print(f"[{nombre}] shape: {df.shape}")
    return df

In [73]:
df_edif = cargar_url_en_df("https://escale.minedu.gob.pe/documents/10156/e36efe68-64b2-4302-b715-0c4b6037d67b", 'Data_edific')
df_aula = cargar_url_en_df("https://escale.minedu.gob.pe/documents/10156/f5035329-93b9-4fd9-bb4c-1e83dcfa91da", 'Data_aula')
df_padron = cargar_url_en_df("https://escale.minedu.gob.pe/documents/10156/aec96875-bdd2-4d3b-a43a-fb7b89d7738c", 'Data_padron')

[Data_edific] Descargado: 3.89 MB | Content-Type: application/zip
[Data_edific] Archivos en ZIP: ['Loc_P53_Edific.dbf']
[Data_edific] shape: (202417, 84)
[Data_aula] Descargado: 6.87 MB | Content-Type: application/zip
[Data_aula] Archivos en ZIP: ['Loc_P61_Aulas.dbf']
[Data_aula] shape: (268839, 96)
[Data_padron] Descargado: 10.72 MB | Content-Type: application/zip
[Data_padron] Archivos en ZIP: ['Padron.dbf']
[Data_padron] shape: (112724, 59)


In [74]:
df_edif.head(3)

,CODLOCAL,NROCED,CUADRO,NUMERO,ID_EDIF,ID_TERR,P53_3,P53_4,P53_5_1,P53_5_2,...,P53_25_3,P53_26_1,P53_26_2,P53_26_3,P53_27_1,P53_27_2,P53_27_3,FEC_ENVIO,GESTION,ANEXO
0,000019,11,C053,1,E01,TE01,1,2,,,...,2024,0,0,,0,0,,2025-09-15 11:02:09,1,0
1,000024,11,C053,1,E01,TE01,1,2,,,...,2025,0,0,,0,0,,2025-09-26 15:04:52,1,0
2,000024,11,C053,2,E02,TE01,1,2,,,...,,0,0,,0,0,,2025-09-26 15:04:52,1,0


In [75]:
df_aula.head(3)

,CODLOCAL,NROCED,CUADRO,NUMERO,ID_EDIF,P61_2,P61_3,P61_4,P61_5_11,P61_5_12,...,P61_22_81,P61_22_82,P61_22_83,P61_22_91,P61_22_92,P61_22_93,FEC_ENVIO,GESTION,ANEXO,AREA_CENSO
0,000019,11,C061,1,E01,1,1,02,A2,3,...,0,0,0,0,1,0,2025-09-15 11:02:09,1,0,1
1,000019,11,C061,2,E01,1,1,02,A2,5,...,0,0,0,0,1,0,2025-09-15 11:02:09,1,0,1
2,000019,11,C061,3,E01,1,1,02,A2,5,...,0,0,0,0,1,0,2025-09-15 11:02:09,1,0,1


In [76]:
df_padron.head(3)

,CODINST,COD_MOD,ANEXO,CODLOCAL,CEN_EDU,NIV_MOD,D_NIV_MOD,D_FORMA,TIPSSEXO,D_TIPSSEXO,...,INTE_VRAEM,TALUM,TALUM_HOM,TALUM_MUJ,TDOC,TSEC,COD_CAR,D_COD_CAR,IMPUTADO,DES_IMPUTA
0,24953981,0415547,0,016100,123,A2,Inicial - Jardín,Escolarizada,3,Mixto,...,No VRAEM,455,218,237,18,16,a,No aplica,1_INFORMANTE,INFOR.MAT_INFOR.DOC
1,25273230,0415638,0,015172,122,A2,Inicial - Jardín,Escolarizada,3,Mixto,...,No VRAEM,433,214,219,20,18,a,No aplica,1_INFORMANTE,INFOR.MAT_INFOR.DOC
2,20414731,0415646,0,015186,233,A2,Inicial - Jardín,Escolarizada,3,Mixto,...,No VRAEM,589,297,292,26,24,a,No aplica,1_INFORMANTE,INFOR.MAT_INFOR.DOC


In [77]:
cols_edif = ['CODLOCAL', 'P53_3', 'P53_4', 'P53_9', 'P53_10', 'P53_13_1', 'P53_13_2', 'P53_13_3']
cols_aula = ['CODLOCAL', 'P61_2', 'P61_3', 'P61_4'] # piso, se usa?, tipo clase
cols_padron = ['COD_MOD', 'CODLOCAL', 'D_FORMA','DAREACENSO','TALUM', 'TDOC', 'D_DPTO']

In [109]:
df_padron2 = df_padron[cols_padron].drop_duplicates(subset=['CODLOCAL'], keep='first')
df_edif2 = df_edif[cols_edif]
df_aula2 = df_aula[cols_aula]

In [90]:
# Conteo de aulas por local
n_aulas = df_aula2.groupby('CODLOCAL').size().rename('n_aulas').reset_index()
# maximo de piso por local
max_piso = df_aula2.groupby('CODLOCAL')['P61_2'].max().rename('max_piso').reset_index()
# Conteo de aulas usadas por local
binary_cols = ['P61_3']
# Suma de 1s en columnas binarias
df_aula2[binary_cols] = (df_aula2[binary_cols] == '1').astype(int)
binary_agg = df_aula2.groupby('CODLOCAL')[binary_cols].sum().reset_index()
# Conteo de cada valor de P53_10 como columnas separadas
P61_4_pivot = (
    df_aula2
    .groupby(['CODLOCAL', 'P61_4'])
    .size()
    .unstack(fill_value=0)
    .add_prefix('P61_4')
).reset_index()
P61_4_pivot = P61_4_pivot[['CODLOCAL', 'P61_401', 'P61_402']]
# Unir todo
df_aula_grouped = n_aulas.merge(max_piso, on='CODLOCAL', how='left').merge(binary_agg, on='CODLOCAL', how='left').merge(P61_4_pivot, on='CODLOCAL', how='left')

In [91]:
binary_cols = ['P53_3', 'P53_4', 'P53_9', 'P53_13_1', 'P53_13_2', 'P53_13_3']
# Conteo de edificios por local
n_edificios = df_edif2.groupby('CODLOCAL').size().rename('n_edificios').reset_index()
# Suma de 1s en columnas binarias
df_edif2[binary_cols] = (df_edif2[binary_cols] == '1').astype(int)
binary_agg = df_edif2.groupby('CODLOCAL')[binary_cols].sum().reset_index()
# Conteo de cada valor de P53_10 como columnas separadas
p53_10_pivot = (
    df_edif2
    .groupby(['CODLOCAL', 'P53_10'])
    .size()
    .unstack(fill_value=0)
    .add_prefix('P5310')
).reset_index()
# Unir todo
df_edif_grouped = n_edificios.merge(binary_agg, on='CODLOCAL', how='left').merge(p53_10_pivot, on='CODLOCAL', how='left')

In [112]:
# dff = df_padron2.merge(df_edif_grouped, on="CODLOCAL", how="left").merge(df_aula_grouped, on="CODLOCAL", how="left")
dff1 = df_edif_grouped.merge(df_aula_grouped, on="CODLOCAL", how="left")
dff2 = df_padron2.merge(dff1, on="CODLOCAL", how="left")
dff2.head(3)

,COD_MOD,CODLOCAL,D_FORMA,DAREACENSO,TALUM,TDOC,D_DPTO,n_edificios,P53_3,P53_4,...,P531008,P531009,P531010,P531011,P531012,n_aulas,max_piso,P61_3,P61_401,P61_402
0,0415547,016100,Escolarizada,Urbana,455,18,ANCASH,4.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,8.0,1.0,0.0,8.0,0.0
1,0415638,015172,Escolarizada,Urbana,433,20,ANCASH,17.0,17.0,0.0,...,0.0,0.0,0.0,0.0,0.0,9.0,1.0,0.0,9.0,0.0
2,0415646,015186,Escolarizada,Urbana,589,26,ANCASH,7.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,12.0,1.0,0.0,0.0,12.0


In [115]:
path_umercea = r"/mnt/c/Users/raini/Downloads/5_URMECEA-HISTÓRICO_lite (1).xlsx"
df_urmecea = pd.read_excel(path_umercea)
df_urmecea.head(3)

,Cod_Modular,Cod_Local,AÑO,Región,Ugel,Provincia,Distrito,Centro_Poblado,Nombre_IE,PDL DIRECTO,...,ACADEMIA,PFC,EPE,Nivel,Área,GRADO,Evaluación,Nivel de logro,PERIODO,UGT
0,358242,81,2024,Ancash,UGEL CORONGO,Corongo,Bambas,BAMBAS,88129 ERNESTO SANCHEZ FAJARDO,NaN,...,NaN,NaN,NaN,Primaria,Rural,4to,LECTURA,En Inicio,ENTRADA,NaN
1,417766,15308,2024,Ancash,UGEL HUARAZ,Huaraz,Huaraz,COILLOR / KEROPAMPA,86003 VIRGEN DE FATIMA,NaN,...,NaN,NaN,NaN,Primaria,Rural,4to,LECTURA,En Proceso,ENTRADA,NaN
2,417766,15308,2024,Ancash,UGEL HUARAZ,Huaraz,Huaraz,COILLOR / KEROPAMPA,86003 VIRGEN DE FATIMA,NaN,...,NaN,NaN,NaN,Primaria,Rural,4to,LECTURA,En Proceso,ENTRADA,NaN


In [154]:
col_urme = ['Cod_Local', 'PDL', 'QM DIRECTO', 'QM INDIRECTO', 'QM', 'PDLD', 'MTESANA', 'ACADEMIA', 'Evaluación', 'Nivel de logro']
df_urmecea_2 = df_urmecea[col_urme]
df_urmecea_2['Cod_Local'] = (
    df_urmecea_2['Cod_Local'].astype('Int64').astype(str).str.zfill(7)
)
map_logro = {
    'en inicio': 1,
    'en proceso': 2,
    'satisfactorio': 3
}

df_urmecea_2['nivel_logro_num'] = df_urmecea_2['Nivel de logro'].str.lower().map(map_logro).astype('Int64')
df_urmecea_2['Evaluación'] = df_urmecea_2['Evaluación'].str.lower()

evaluacion_pivot = (
    df_urmecea_2
    .groupby(['Cod_Local', 'Evaluación'])
    .agg({'nivel_logro_num': 'mean'})
    .unstack(fill_value=0)
    .add_prefix('Evaluación')
).reset_index()
evaluacion_pivot.columns = ['Cod_Local', 'Evaluación_lectura', 'Evaluación_matematica']

cols_binarias = ['PDL', 'QM DIRECTO', 'QM INDIRECTO', 'QM', 'PDLD', 'MTESANA', 'ACADEMIA']
df_urmecea_2[cols_binarias] = df_urmecea_2[cols_binarias].notna().astype(int)
df_urmecea_3 = df_urmecea_2[['Cod_Local'] + cols_binarias].rename(columns={
    'MTESANA': 'MTESANA_INDIRECTO',
    'ACADEMIA': 'ACADEMIA_DIRECTO'
})
df_urmecea_3.drop_duplicates(subset=['Cod_Local'], keep='first', inplace=True)

df_urmecea_4 = df_urmecea_3.merge(evaluacion_pivot, on='Cod_Local', how='left').rename(columns={'Cod_Local': 'CODLOCAL'})

df_urmecea_4.head(3)

,CODLOCAL,PDL,QM DIRECTO,QM INDIRECTO,QM,PDLD,MTESANA_INDIRECTO,ACADEMIA_DIRECTO,Evaluación_lectura,Evaluación_matematica
0,0000081,0,0,0,0,0,0,0,1.0,1.0
1,0015308,0,0,0,0,0,0,0,1.857143,1.0
2,0015313,0,0,0,0,0,0,0,1.823529,1.25


In [155]:
df_final = df_urmecea_4.merge(dff2, on='CODLOCAL', how='left')

In [159]:
df_final

,CODLOCAL,PDL,QM DIRECTO,QM INDIRECTO,QM,PDLD,MTESANA_INDIRECTO,ACADEMIA_DIRECTO,Evaluación_lectura,Evaluación_matematica,...,P531008,P531009,P531010,P531011,P531012,n_aulas,max_piso,P61_3,P61_401,P61_402
0,0000081,0,0,0,0,0,0,0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0015308,0,0,0,0,0,0,0,1.857143,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0015313,0,0,0,0,0,0,0,1.823529,1.25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0015327,0,0,0,0,0,0,0,1.8,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0015332,0,0,0,0,0,0,0,1.477273,1.071429,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
931,0131652,0,0,0,0,0,0,0,0.0,1.647059,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
932,0131770,0,0,0,0,0,0,0,0.0,1.867925,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
933,0819393,0,0,0,0,0,0,0,1.763158,1.457143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
934,0131789,0,0,0,0,0,0,0,1.910112,1.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
